# AI for Coding and Data Analysis: starter notebook

Workshop, October 14, 2026 · Digital Scholarship and Data Services, Sheridan Libraries

**Before you start:** File → *Save a copy in Drive*, then add your name to the title. You'll work in your own copy.

**About the data:** a *synthetic* export from a fictional "Research Data Practices" survey, built for teaching.

**How to use this notebook:** each section has prompts to try with Gemini. Triple-click a prompt to select, copy, and paste it into Gemini (the ✦ button or the Gemini panel). Run cells with ▶ or Shift+Enter.

## 1. Get the data
Run the cell below to download the survey exports and the codebook from GitHub into this session's temporary storage (see the 📁 Files panel on the left).

The files will disappear when the notebook is closed or reloaded. They can be downloaded again from GitHub or the code can be modified to work with local files.

The end of the cell holds the **settings**: which export to analyze and how complete a response must be to keep it. Decisions like these live in one place, at the top, so they're easy to see and to change.

In [ ]:
# Download the survey exports and codebook from GitHub into this environment.
import urllib.request

BASE = "https://raw.githubusercontent.com/jhu-data-services/ai-coding-data-analysis/main/data/"
FILES = ["data_practices_survey_2026-09-30.xlsx",
         "data_practices_survey_2026-10-09.xlsx",
         "data_practices_survey_codebook.xlsx"]

for f in FILES:
    urllib.request.urlretrieve(BASE + f, f)
    print("downloaded", f)

# Settings: the choices the rest of the notebook depends on.
# Change the name of the file here to analyze a different file.
DATA_FILE = "data_practices_survey_2026-09-30.xlsx"
MIN_PROGRESS = 50   # keep responses at least this % complete; partial responses past 50% answered the sharing questions
print("analyzing", DATA_FILE, "| keeping responses at least", MIN_PROGRESS, "% complete")

## 2. Load and look
Look at the data before you change anything.

**Prompt to try**:

> The variable DATA_FILE holds the name of a survey file exported from Qualtrics. Row 1 has the column names; rows 2 and 3 hold question text and import IDs and need to be skipped. Load the file with pandas into a DataFrame called df_raw. Show the number of rows, the column names, and the first 5 rows. Don't modify the file.

**Then try**:

> For each column in df_raw, show its type and how many values are missing. List the unique values of Status, Progress, and Finished.

**Then check the codebook**:

> Load data_practices_survey_codebook.xlsx and show it as a table. Which questions allow several answers, which are free text, and which are skipped for some respondents?

## 3. Clean, one step at a time
Ask for one step per cell, and ask Gemini to explain what it wrote. Above each cleaning step, add a text cell stating the rule and why you chose it. Steps we'll take in the demo:
1. Keep only non-preview responses that are complete enough (`Status` is "IP Address" and `Progress` is at least `MIN_PROGRESS`, set in the first code cell).
2. Remove duplicate submissions: same IP address and same answers. (**Note**: `ResponseId` only can't catch these, because every submission gets a new one.)
3. Drop identifying and empty metadata columns (IP address, location, recipient fields). Do this after step 2, which needs the IP address.
4. Turn the satisfaction labels in `Q7` into numbers 1-5 in a new column called `satisfaction`.
5. Turn the free-text hours in `Q4` into numbers in a new column called `hours`, using a rule you write down.

Call the cleaned DataFrame `df`.

### Verify the cleaning
Before you analyze anything, check that each cleaning step did what you intended. If a check fails and you can't fix it, run the catch-up cell below.

**Prompts to try, one at a time**:

> Write checks that compare the cleaned DataFrame df with df_raw, explain why these check were written.

> Write checks for the recoded columns and explain why these checks would work.

**Then look for yourself:** do the row counts make sense? Does hours conversion follow your rule? Pick two rows in df_raw and find them in df: did they change the way you expected?

### Catch-up cell
If your cleaning went sideways, set `USE_CATCH_UP = True` in the cell below and run it. It makes the same decisions as the demo and gives you a clean DataFrame called `df`, so you can continue with the next sections.

Leave it `False` otherwise, so that **Restart and run all** checks *your* cleaning, not this one.

In [ ]:
USE_CATCH_UP = False   # set to True only if your own cleaning didn't work

import re
import pandas as pd

# Qualtrics exports the satisfaction answers as labels; map them to 1-5 (higher = more satisfied).
SATISFACTION = {"Extremely dissatisfied": 1, "Somewhat dissatisfied": 2,
                "Neither satisfied nor dissatisfied": 3,
                "Somewhat satisfied": 4, "Extremely satisfied": 5}

def hours_to_number(text):
    """Documented rule: '5' -> 5; '4-6' -> 5 (midpoint); '10+' -> 10; 'less than 1' -> 0.5;
    '~5', 'about 5', '5ish', '5 hrs' -> 5; answers with no number ('depends') -> missing."""
    if pd.isna(text):
        return None
    t = str(text).strip().lower()
    if t in ("less than 1", "<1"):
        return 0.5
    m = re.fullmatch(r"(\d+)\s*-\s*(\d+)", t)
    if m:
        return (int(m[1]) + int(m[2])) / 2
    m = re.search(r"\d+(\.\d+)?", t)
    return float(m[0]) if m else None

def clean_export(path):
    raw = pd.read_excel(path, skiprows=[1, 2])                  # skip question-text and import-ID rows
    df = raw[(raw["Status"] == "IP Address") & (raw["Progress"] >= MIN_PROGRESS)]   # drop previews and responses below the completeness threshold
    answers = [c for c in df.columns if c.startswith("Q")]
    # ResponseId is new for every submission, so a repeat submission is matched on IP address + identical answers.
    df = df.drop_duplicates(subset=["IPAddress"] + answers)
    # Drop identifying columns (IP, location) and recipient fields that are always empty for an anonymous link.
    df = df.drop(columns=["IPAddress", "LocationLatitude", "LocationLongitude",
                          "RecipientLastName", "RecipientFirstName", "RecipientEmail",
                          "ExternalReference"])
    df["hours"] = df["Q4"].map(hours_to_number)
    df["satisfaction"] = df["Q7"].map(SATISFACTION)
    return df.reset_index(drop=True)                            # renumber rows 0..n-1 after filtering

if USE_CATCH_UP:
    df = clean_export(DATA_FILE)
    print(len(df), "valid responses (catch-up cleaning)")
else:
    print("Catch-up skipped: using your own df.")

## 4a. Q5: What kinds of sharing?
Question `Q5` allowed several answers, joined with semicolons (for example `Within my research team;By request`). We need to separate them for analysis.

**Prompt to try**:

> In df, the column Q5 holds multiple answers separated by semicolons. Make a table called q5_long with one row per respondent and option (columns ResponseId and option), then a table called q5_counts with how many respondents chose each option.

**Check your result**:

> I need to verify that the split and counting was done correctly. Suggest a way.


## 4b. Q6: How do people share data?
`Q6` is also multi-answer, joined with semicolons. Respondents who chose "No sharing" in `Q5` never saw it, and "Other" answers are in `Q6_9_TEXT`.

## 5. Summarize and chart

**Prompts to try, one at a time** (triple-click to select, then copy into Gemini):

> Make a table called by_discipline with the number of respondents, mean hours, and mean satisfaction by discipline (Q1).

> Make a bar chart of mean hours by primary tool (Q3).

> Sort the bars, label the axes, and use colorblind-friendly colors. Save the final chart as a PNG in a folder called outputs (create it if needed).

**Check the table a different way:**

> Filter df to one discipline and calculate its number of respondents and mean hours directly, without by_discipline. Confirm they match the table. Then confirm that the respondent counts in by_discipline add up to the number of rows in df.

## 6. Themes in the comments: AI reads, you decide
`Q8` holds optional free-text comments. Code alone can't read what a comment means, so the cells below send the comments to a language model through Colab's built-in `google.colab.ai` library (free, no API key) and bring its answers back into the notebook.

We work in four steps, the way researchers use AI for qualitative coding:
1. **The AI proposes themes** after reading all the comments.
2. **You decide the codebook**: the final themes and their definitions.
3. **The AI applies your codebook** to every comment and gives a reason for each choice.
4. **You check it** by coding some comments yourself and comparing.

The data here is synthetic. With real data, check that you're allowed to send it to an AI service before you run these cells.

In [ ]:
# 6.1 Collect the comments and choose the model.
import os
import pandas as pd
from google.colab import ai

MODEL = "google/gemini-3.5-flash"   # record which model did the reading; ai.list_models() shows the options
comments = df.loc[df["Q8"].notna(), ["ResponseId", "Q8"]].rename(columns={"Q8": "comment"})
os.makedirs("outputs", exist_ok=True)
print(len(comments), "comments")

### Step 1: the AI proposes themes
The prompt is written out in the cell so it's part of the record. Change it if you want different guidance.

In [ ]:
# 6.2 The model reads all comments and proposes themes.
PROPOSE_PROMPT = """You are helping with a qualitative analysis of open-ended survey comments
about how researchers manage and share their research data.
Read all the comments below and propose 6 to 8 themes.
For each theme give a short name, a one-sentence definition, and two example quotes copied exactly from the comments.
Do not invent quotes. Treat non-answers such as "n/a" or "thanks" as a separate group.

Comments:
"""
proposal = ai.generate_text(PROPOSE_PROMPT + "\n".join("- " + c for c in comments["comment"]), model_name=MODEL)
open("outputs/theme_proposal.txt", "w").write(proposal)   # keep what the model proposed
print(proposal)

### Step 2: you decide the codebook
Read the proposal above, then edit the themes below: merge, rename, add or remove. This is an interpretive decision, so it lives in the notebook, like `MIN_PROGRESS`. The starting list is the facilitator's codebook; make it yours.

In [ ]:
# 6.3 YOUR codebook: theme name -> definition.
CODEBOOK = {
    "Cleaning burden": "Time and effort spent cleaning, reformatting, or merging data.",
    "Storage and backup": "Where data is stored, backed up, or kept long term.",
    "Versions and documentation": "Tracking file versions, naming, and documenting what was done.",
    "Training and support": "Wishes for workshops, courses, or one-on-one help.",
    "Barriers to sharing": "Reasons people don't share data, or uncertainty about how to share.",
    "Sensitive data and compliance": "Privacy, IRB, HIPAA, de-identification, or which tools are allowed.",
    "Funder and journal requirements": "Data management plans, sharing mandates, availability statements.",
    "Tools and software": "Problems with or preferences for specific software.",
    "No substantive comment": "Non-answers such as 'n/a', 'no', or 'thanks'.",
}

### Step 3: the AI applies your codebook
Each comment gets one theme from your codebook and a one-sentence reason. The result is the table `comment_themes`.

In [ ]:
# 6.4 The model assigns each comment to a theme in YOUR codebook and explains why.
import json, re
from datetime import date

ASSIGN_PROMPT = """Assign each survey comment below to exactly one theme from this codebook.
Use only the theme names given. If no theme fits, use "Other".
Return only JSON: a list of objects with the keys "id", "theme", and "reason" (one short sentence).

Codebook:
{codebook}

Comments:
{comments}"""
prompt = ASSIGN_PROMPT.format(
    codebook="\n".join(f"- {name}: {definition}" for name, definition in CODEBOOK.items()),
    comments="\n".join(f"{row.ResponseId}: {row.comment}" for row in comments.itertuples()),
)
reply = ai.generate_text(prompt, model_name=MODEL)

found = re.search(r"\[.*\]", reply, re.DOTALL)   # keep only the JSON list, in case the model adds text around it
assignments = pd.DataFrame(json.loads(found.group(0))).rename(columns={"id": "ResponseId"})
comment_themes = comments.merge(assignments, on="ResponseId", how="left")

# Quick checks: every comment got a theme, and only codebook themes were used.
missing = comment_themes["theme"].isna().sum()
unknown = sorted(set(comment_themes["theme"].dropna()) - set(CODEBOOK) - {"Other"})
print("comments without a theme:", missing, "| themes not in the codebook:", unknown or "none")

# Record how this coding was produced: model, date, codebook, and prompt.
with open("outputs/theme_coding_record.txt", "w") as f:
    f.write(f"Model: {MODEL}\nDate: {date.today()}\nData: {DATA_FILE}\n\nCodebook:\n")
    f.write("\n".join(f"- {k}: {v}" for k, v in CODEBOOK.items()))
    f.write("\n\nPrompt template:\n" + ASSIGN_PROMPT)

comment_themes["theme"].value_counts()

### Step 4: check it yourself
Code a sample **before** you look at the AI's themes. Everyone gets the same 10 comments.

In [ ]:
# 6.5 Ten comments to code yourself (fixed sample, so everyone sees the same ones).
sample = comments.sample(10, random_state=1).reset_index(drop=True)
sample

In [ ]:
# 6.6 Type a theme from your codebook for each comment above, in the same order, then compare with the AI.
my_themes = ["", "", "", "", "", "", "", "", "", ""]

compare = sample.assign(my_theme=my_themes).merge(
    comment_themes[["ResponseId", "theme", "reason"]], on="ResponseId", how="left")
compare["agree"] = compare["my_theme"] == compare["theme"]
print(f"You and the AI agree on {compare['agree'].sum()} of {len(compare)} comments.")
compare

Where you disagree, read the AI's reason. Who's right, or is the codebook unclear?

Then run step 3 again. Do some assignments change? That's why you save the version you report: section 10 saves `comment_themes`, the theme proposal, and the coding record.

## 7. Check: restart and run all
**Runtime → Restart session and run all.** When it finishes, this cell should report the numbers below with no errors.

In [ ]:
# Known values for each export (with MIN_PROGRESS = 50)
EXPECTED = {"data_practices_survey_2026-09-30.xlsx": 152,
            "data_practices_survey_2026-10-09.xlsx": 213}

print("File:", DATA_FILE)
print("Valid responses:", len(df))
if "hours" in df.columns:
    print("Mean hours per week cleaning data:", round(df["hours"].mean(), 1))
assert len(df) == EXPECTED[DATA_FILE], (
    f"Expected {EXPECTED[DATA_FILE]} valid responses: check the filtering and duplicate steps.")
print("Checks passed.")

## 8. A new export arrives
The survey stayed open. `data_practices_survey_2026-10-09.xlsx` contains every earlier response plus the new ones.

1. In the first code cell, change `DATA_FILE` to `"data_practices_survey_2026-10-09.xlsx"`.
2. **Runtime → Restart session and run all.**

Every table and chart updates, and the check cell should now report 213 valid responses. No new code: that's the payoff of writing the steps down.

## 9. Document and share
**Prompt to try** (triple-click to select, then copy into Gemini):

> Add a text cell above each code cell explaining in plain language what it does and why. At the top, add a header with the data source, date, tools used, AI assistance, how to run this notebook, and a short list of the cleaning decisions and the reasons for them.

Read what it writes and correct anything that misstates a decision.

## 10. Save your outputs
Tables and charts live in memory and disappear when the runtime resets. The code is what lasts: **Run all** rebuilds everything. To keep or share results, save them as files.

The cell below saves every table you created (except the raw data) to a folder named after the export (for example `outputs/2026-09-30`), with the export date in each file name, and zips that folder. Download the zip from the 📁 Files panel (right-click → Download). Charts are saved if your chart code saved them into `outputs`.

Never edit an output file by hand. If something needs to change, change the code and run everything again.

In [ ]:
# Save results as files, named after the export they came from.
import os, shutil
import pandas as pd

tag = DATA_FILE.replace("data_practices_survey_", "").replace(".xlsx", "")   # e.g. 2026-09-30
OUT = f"outputs/{tag}"                     # one folder per export, so results never mix
os.makedirs(OUT, exist_ok=True)

# Every table (DataFrame) created in this notebook, except the raw export, which is already saved.
for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame) and not name.startswith("_") and name != "df_raw":
        path = f"{OUT}/{name}_{tag}.csv"
        obj.to_csv(path, index=False)
        print("saved", path)

# Charts and text records saved directly into outputs/ are moved into this export's folder.
for f in os.listdir("outputs"):
    if f.lower().endswith((".png", ".jpg", ".svg", ".pdf", ".txt")):
        base, ext = os.path.splitext(f)
        shutil.move(f"outputs/{f}", f"{OUT}/{base}_{tag}{ext}")
        print("saved", f"{OUT}/{base}_{tag}{ext}")

shutil.make_archive(f"outputs_{tag}", "zip", OUT)
print(f"Download outputs_{tag}.zip from the Files panel.")

# Optional: copy the outputs to Google Drive instead (Colab will ask for permission).
# from google.colab import drive
# drive.mount("/content/drive")
# shutil.copytree(OUT, f"/content/drive/MyDrive/{OUT}", dirs_exist_ok=True)